In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

# Load the raw collected data (saved by 01_data_collection.ipynb)
raw_files = sorted(Path("../data/raw").glob("adzuna_raw_*.json"))
latest_file = raw_files[-1]
print(f"Loading: {latest_file.name}")

with open(latest_file, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

df = pd.DataFrame(raw_data)
print(f"Loaded: {len(df):,} raw postings, {len(df.columns)} columns")

Loading: adzuna_raw_20260603_140710.json
Loaded: 3,653 raw postings, 19 columns


In [3]:
print(f"Total rows: {len(df)}")
print(f"Unique IDs: {df['id'].nunique()}")
print(f"Duplicates by ID: {len(df) - df['id'].nunique()}")

Total rows: 3653
Unique IDs: 3332
Duplicates by ID: 321


In [4]:
print("Column profile:\n")

rows = []
for col in df.columns:
    dtype = str(df[col].dtype)
    non_null = df[col].notna().sum()
    null_pct = round(df[col].isna().sum() / len(df) * 100, 2)
    
    sample = df[col].dropna().iloc[0] if df[col].notna().any() else "—"
    sample_str = str(sample)[:60]
    
    if dtype == "object":
        try:
            n_unique = df[col].nunique()
        except TypeError:
            n_unique = "(nested - dict)"
    else:
        n_unique = df[col].nunique()
    
    rows.append([col, dtype, non_null, null_pct, n_unique, sample_str])

profile = pd.DataFrame(rows, columns=["column", "dtype", "non_null", "null_pct", "unique_values", "sample"])
print(profile.to_string(index=False))

Column profile:

             column   dtype  non_null  null_pct   unique_values                                                       sample
            company  object      3653      0.00 (nested - dict) {'display_name': 'Fieldfisher', '__CLASS__': 'Adzuna::API::R
       redirect_url     str      3653      0.00            3449 https://www.adzuna.co.uk/jobs/land/ad/5730148926?se=Sl_UdExf
salary_is_predicted     str      3653      0.00               2                                                            1
           category  object      3653      0.00 (nested - dict) {'tag': 'it-jobs', 'label': 'IT Jobs', '__CLASS__': 'Adzuna:
         salary_min float64      3652      0.03            2165                                                     39567.07
           location  object      3653      0.00 (nested - dict) {'display_name': 'Belfast, Northern Ireland', '__CLASS__': '
              title     str      3653      0.00            1825                                             

In [5]:
print("Salary data quality:")
print(df["salary_is_predicted"].value_counts())
print()

real_salaries = df[df["salary_is_predicted"] == "0"]["salary_min"]
pred_salaries = df[df["salary_is_predicted"] == "1"]["salary_min"]

print(f"Real (employer-posted) salaries: {len(real_salaries):,}")
print(f"  Mean: £{real_salaries.mean():,.0f}, Median: £{real_salaries.median():,.0f}")
print()
print(f"Predicted (Adzuna estimate) salaries: {len(pred_salaries):,}")
print(f"  Mean: £{pred_salaries.mean():,.0f}, Median: £{pred_salaries.median():,.0f}")

Salary data quality:
salary_is_predicted
1    2143
0    1510
Name: count, dtype: int64

Real (employer-posted) salaries: 1,510
  Mean: £56,380, Median: £49,920

Predicted (Adzuna estimate) salaries: 2,143
  Mean: £57,718, Median: £55,980


In [6]:
df["category_label"] = df["category"].apply(lambda x: x.get("label") if isinstance(x, dict) else None)
print("Category distribution:")
print(df["category_label"].value_counts())

Category distribution:
category_label
IT Jobs                             2618
Accounting & Finance Jobs            284
Engineering Jobs                     268
Scientific & QA Jobs                  79
Sales Jobs                            60
Consultancy Jobs                      56
PR, Advertising & Marketing Jobs      52
Teaching Jobs                         48
Graduate Jobs                         34
Admin Jobs                            29
HR & Recruitment Jobs                 22
Healthcare & Nursing Jobs             19
Social work Jobs                      16
Logistics & Warehouse Jobs            15
Trade & Construction Jobs             12
Energy, Oil & Gas Jobs                11
Retail Jobs                            9
Part time Jobs                         7
Legal Jobs                             3
Manufacturing Jobs                     3
Creative & Design Jobs                 3
Customer Services Jobs                 2
Hospitality & Catering Jobs            1
Maintenance Jobs   

In [7]:
clean = df.copy()

# Flatten nested dicts
clean["company_name"] = clean["company"].apply(
    lambda x: x.get("display_name") if isinstance(x, dict) else None
)
clean["location_full"] = clean["location"].apply(
    lambda x: x.get("display_name") if isinstance(x, dict) else None
)
clean["location_area"] = clean["location"].apply(
    lambda x: x.get("area") if isinstance(x, dict) else None
)
clean["category_label"] = clean["category"].apply(
    lambda x: x.get("label") if isinstance(x, dict) else None
)
clean["category_tag"] = clean["category"].apply(
    lambda x: x.get("tag") if isinstance(x, dict) else None
)

# Aggregate search keywords before dedup
keywords_per_id = clean.groupby("id")["_search_keyword"].agg(
    lambda s: ", ".join(sorted(set(s)))
).reset_index()
keywords_per_id.columns = ["id", "found_under_keywords"]

# Deduplicate by id (keep first occurrence)
before = len(clean)
clean = clean.drop_duplicates(subset="id", keep="first")
print(f"Deduplication: {before} -> {len(clean)} ({before - len(clean)} duplicates removed)")

# Merge in the aggregated keywords
clean = clean.merge(keywords_per_id, on="id", how="left")
clean["n_keywords_matched"] = clean["found_under_keywords"].str.count(",") + 1

# Drop useless columns
drop_cols = ["__CLASS__", "adref", "redirect_url", "company", "location", "category", "_search_keyword"]
clean = clean.drop(columns=[c for c in drop_cols if c in clean.columns])

# Derived features
clean["title_length"] = clean["title"].str.len()
clean["description_length"] = clean["description"].str.len()
clean["description_word_count"] = clean["description"].str.split().str.len()
clean["salary_midpoint"] = (clean["salary_min"] + clean["salary_max"]) / 2
clean["salary_is_predicted"] = clean["salary_is_predicted"].astype(int).astype(bool)
clean["created"] = pd.to_datetime(clean["created"])

print(f"\nClean dataset shape: {clean.shape}")

Deduplication: 3653 -> 3332 (321 duplicates removed)

Clean dataset shape: (3332, 23)


In [8]:
print("Description length distribution:")
print(clean["description_word_count"].describe().round(1))
print()

counts, bins = np.histogram(clean["description_word_count"], bins=[0, 10, 25, 50, 100, 200, 500, 1000, 5000])
print("Word count buckets:")
for i in range(len(counts)):
    bar = "█" * int(counts[i] / 30)
    print(f"  {bins[i]:>4}-{bins[i+1]:<4}: {counts[i]:>4} {bar}")

print()

# Sample to confirm truncation
print("Last 80 chars of 5 sample descriptions:")
for i in range(5):
    desc = clean.iloc[i]['description']
    print(f"  {i}: ...{desc[-80:]}")

Description length distribution:
count    3332.0
mean       75.6
std         6.1
min        22.0
25%        72.0
50%        76.0
75%        80.0
max        94.0
Name: description_word_count, dtype: float64

Word count buckets:
     0-10  :    0 
    10-25  :    1 
    25-50  :    2 
    50-100 : 3329 ██████████████████████████████████████████████████████████████████████████████████████████████████████████████
   100-200 :    0 
   200-500 :    0 
   500-1000:    0 
  1000-5000:    0 

Last 80 chars of 5 sample descriptions:
  0: ...Senior Data Engineer and sits within the Cloud and Infrastructure team, reporti…
  1: ...space company who are constantly growing and developing. They are always lookin…
  2: ...over every meal occasion from breakfast through to dinner and dessert, with lun…
  3: ...ies As our Business Intelligence Analyst, you will be responsible for a blend o…
  4: ...ies As our Business Intelligence Analyst, you will be responsible for a blend o…


In [9]:
print(f"Total unique titles: {clean['title'].nunique()} across {len(clean)} postings\n")

print("Top 20 most common titles:")
print(clean['title'].value_counts().head(20).to_string())
print()

print(f"\nTitles appearing only once: {(clean['title'].value_counts() == 1).sum()}")
print(f"Titles appearing 2-5 times: {((clean['title'].value_counts() >= 2) & (clean['title'].value_counts() <= 5)).sum()}")
print(f"Titles appearing 6+ times: {(clean['title'].value_counts() >= 6).sum()}")

Total unique titles: 1825 across 3332 postings

Top 20 most common titles:
title
Data Analyst                                             138
Data Science Trainee                                     109
Data Scientist                                           109
Data Analyst Trainee                                      70
Senior Data Scientist                                     49
Senior Data Analyst                                       45
Data Engineer                                             37
Analytics Engineer                                        28
Lead Data Scientist                                       27
Senior Analytics Engineer                                 24
Power BI Developer                                        23
Trainee Data Analyst                                      22
Senior Data Engineer                                      21
Business Intelligence Analyst                             19
Employment Adviser                                        16
Data

In [10]:
trainee_postings = clean[clean['title'].str.contains('trainee', case=False, na=False)]
print(f"Trainee-titled postings: {len(trainee_postings)}")
print(f"\nUnique companies posting these:")
print(trainee_postings['company_name'].value_counts().head(10).to_string())
print()
print("Sample descriptions (first 2):")
for i, row in trainee_postings.head(2).iterrows():
    print(f"\n[{row['title']} - {row['company_name']}]")
    print(f"  {row['description'][:300]}")

Trainee-titled postings: 212

Unique companies posting these:
company_name
IT Career Switch                      104
ITOL Recruit                           95
Newto Training                          4
Future Engineering                      2
CLC Utility Services Ltd                1
Ecruit                                  1
IT Online Learning                      1
Heart of Yorkshire Education Group      1
Cawood Scientific                       1
Future Engineering Recruitment Ltd      1

Sample descriptions (first 2):

[Data Analyst Trainee - ITOL Recruit]
  Are you looking to benefit from a new career in Data Analysis? If you are detail orientated, perceptive, organised, competent, analytical and can communicate well with those around you; you could have a truly rewarding future as a Data Analyst Demand for Data Analysts has grown 20% year on year with

[Data Analyst Trainee - ITOL Recruit]
  Are you looking to benefit from a new career in Data Analysis? If you are detail orientate

In [11]:
noise_titles = ['Employment Adviser', 'Principal Fire Engineer']
print("Junk samples:\n")
for title in noise_titles:
    samples = clean[clean['title'] == title]
    print(f"=== {title} ({len(samples)} postings) ===")
    print(f"Categories: {samples['category_label'].value_counts().to_dict()}")
    print(f"Found under keywords: {samples['found_under_keywords'].value_counts().to_dict()}")
    print(f"\nSample description:")
    print(f"  {samples.iloc[0]['description'][:300]}")
    print()

Junk samples:

=== Employment Adviser (16 postings) ===
Categories: {'Teaching Jobs': 12, 'Social work Jobs': 4}
Found under keywords: {'BI developer': 16}

Sample description:
  Are you passionate about making a difference and helping others to fulfil their potential? Would you like to work in a role that puts the customer at the heart of everything we do, making a genuine positive impact? Then consider the role of Employment Adviser at Reed in Partnership! Who we are: Reed

=== Principal Fire Engineer (12 postings) ===
Categories: {'Engineering Jobs': 12}
Found under keywords: {'analytics engineer': 12}

Sample description:
  Job Description Start here. Grow here. AECOM is seeking a Principal Fire Engineer to join our Fire Team. This consultancy role requires proven experience across various sectors and technical fire engineering applications in a client-focused environment. In this role, you will work closely with offic



In [12]:
print("=== Companies posting >=10 ads ===")
print(clean['company_name'].value_counts().head(20).to_string())
print()

print("=== Non-IT postings by category and top titles ===")
non_it = clean[clean['category_label'] != 'IT Jobs']
print(f"Total non-IT: {len(non_it)} ({len(non_it)/len(clean)*100:.1f}%)")
print()

for cat in non_it['category_label'].value_counts().head(5).index:
    print(f"\n--- {cat} ---")
    print(non_it[non_it['category_label']==cat]['title'].value_counts().head(5).to_string())

=== Companies posting >=10 ads ===
company_name
Harnham - Data & Analytics Recruitment    104
IT Career Switch                          104
ITOL Recruit                               95
Reed                                       33
Reed Talent Solutions                      23
Amazon                                     22
IT Online Learning                         21
MCS Group                                  19
Adecco                                     18
Pro Contract Jobs Ltd                      18
Fresha                                     17
Ocho                                       16
IBM                                        16
Spotify                                    15
ey                                         15
Data Idols                                 13
JPMorganChase                              13
Preply                                     13
Harnham                                    13
Anson Mccade                               12

=== Non-IT postings by category

In [13]:
def is_junk(row):
    course_companies = ["IT Career Switch", "ITOL Recruit", "IT Online Learning"]
    if row["company_name"] in course_companies:
        return True
    
    if row["category_label"] == "Engineering Jobs":
        title_lower = str(row["title"]).lower()
        if any(term in title_lower for term in ["analytics", "data", "analyst", "scientist", "business intelligence", "bi developer"]):
            return False
        return True
    
    if row["category_label"] == "Sales Jobs":
        title_lower = str(row["title"]).lower()
        if any(term in title_lower for term in ["analytics", "data scientist", "data analyst"]):
            return False
        return True
    
    noise_titles = ["Employment Adviser", "Principal Fire Engineer"]
    if row["title"] in noise_titles:
        return True
    
    return False

clean["is_junk"] = clean.apply(is_junk, axis=1)
print(f"Junk identified: {clean['is_junk'].sum():,} of {len(clean):,} ({clean['is_junk'].sum()/len(clean)*100:.1f}%)")
print()
print("Breakdown:")
print(clean[clean["is_junk"]]["company_name"].value_counts().head(10).to_string())
print()
print(clean[clean["is_junk"]]["category_label"].value_counts().to_string())

Junk identified: 497 of 3,332 (14.9%)

Breakdown:
company_name
IT Career Switch         104
ITOL Recruit              95
IT Online Learning        21
Reed Talent Solutions     15
Fresha                    14
Hire Resolve.com          11
Reed                      10
AECOM                      9
Morson Edge                5
AKT II                     4

category_label
IT Jobs             220
Engineering Jobs    204
Sales Jobs           57
Teaching Jobs        12
Social work Jobs      4


In [14]:
junk_summary = {
    "total_before_filter": len(clean),
    "removed_course_adverts": ((clean["is_junk"]) & (clean["company_name"].isin(["IT Career Switch", "ITOL Recruit", "IT Online Learning"]))).sum(),
    "removed_engineering_noise": ((clean["is_junk"]) & (clean["category_label"] == "Engineering Jobs")).sum(),
    "removed_sales_noise": ((clean["is_junk"]) & (clean["category_label"] == "Sales Jobs")).sum(),
    "removed_other_noise": ((clean["is_junk"]) & (~clean["category_label"].isin(["Engineering Jobs", "Sales Jobs"])) & (~clean["company_name"].isin(["IT Career Switch", "ITOL Recruit", "IT Online Learning"]))).sum(),
    "total_removed": clean["is_junk"].sum(),
    "remaining": (~clean["is_junk"]).sum()
}

print("Junk filter summary (for README narrative):")
for key, value in junk_summary.items():
    print(f"  {key}: {value:,}")

# Apply filter
final = clean[~clean["is_junk"]].copy()
final = final.drop(columns=["is_junk"])

print(f"\nFinal dataset: {len(final):,} postings")
print(f"  Categories: {final['category_label'].nunique()}")
print(f"  Companies: {final['company_name'].nunique()}")
print(f"  Unique titles: {final['title'].nunique()}")

# Save
output_file = Path("../data/processed/postings_clean.parquet")
output_file.parent.mkdir(parents=True, exist_ok=True)
final.to_parquet(output_file, index=False)
print(f"\nSaved: {output_file}")
print(f"File size: {output_file.stat().st_size / 1024:.1f} KB")

Junk filter summary (for README narrative):
  total_before_filter: 3,332
  removed_course_adverts: 220
  removed_engineering_noise: 204
  removed_sales_noise: 57
  removed_other_noise: 16
  total_removed: 497
  remaining: 2,835

Final dataset: 2,835 postings
  Categories: 25
  Companies: 1412
  Unique titles: 1658

Saved: ..\data\processed\postings_clean.parquet
File size: 819.6 KB
